## Data: from df to cleaned df (for each task)

In [ ]:
task = "task8"
_pca = "True"
pca_truncation = "False"

### can choose to perform PCA

In [ ]:
import numpy as np 
import pandas as pd 
from matplotlib import pyplot as plt
import os
import random
import scipy
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.utils import shuffle
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
excel_path = "data/new_hypoF_PCV 220919.xlsx"
task_dict = { # target and features to be dropped
    "task1" : ["group", ["injection_demand" , "vision_gain", "rec_period"]], # binary
    "task2" : ["injection_demand", ["total_inj_n", "group", "vision_gain", "rec_period"]], # binary
    # "task3" : ["total_inj_n", ["injection_demand", "group", "vision_gain", "rec_period"]], # regression
    # "task4" : ["rec_period", ["group", "vision_gain", "injection_demand", "recur"]],# regression
    # "task5" : ["recur", ["group", "vision_gain", "injection_demand", "rec_period"]], # binary
    # "task6" : ["vision_gain",["group", "injection_demand", "rec_period"]], # binary
    "task8" : ["time_to_rem", ["group", "rec_period", "total_inj_n", "vision_gain", "injection_demand", "recur"]],# regression
    
    }


seed = 2022
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
df = pd.read_excel(excel_path, sheet_name="cleaned_0")
df = df.rename(columns={"HypoF": "ASHS-LIA", "basline_logMAR": "baseline_logMAR", "largest_polyp": "Largest polyp diameter", "number_of_polyps": "Polyp number", "lipid": "Lipid exudation", "CNV_location": "Polyp location", "SRHeme": "SRH"})
for col in df.columns:
    print(col)
df.head()


In [ ]:
import numpy as np
import pandas as pd
from pca import pca

df_y, df_x = df[task_dict[task][0]], df.drop(columns = task_dict[task][1])
# df_x = df_x.drop(columns = ["fu_period", "recur"])
df_x = df_x.drop(columns = task_dict[task][0])
if task == "task1":
    df_y = df_y - 1
    
df_b4_standardization = pd.concat([df_y, df_x], axis=1, join='inner')
df_b4_standardization.dropna(inplace=True)
# int --> float
for col in df_b4_standardization.columns:
    df_b4_standardization[col] = df_b4_standardization[col].astype(float)
# standardization on x
cols = df_b4_standardization.columns
cols_to_be_standardized = cols[1:]
standard_scaler = StandardScaler()
df_b4_standardization[cols_to_be_standardized] = standard_scaler.fit_transform(df_b4_standardization[cols_to_be_standardized])
#
df_standardized = df_b4_standardization
print(df_standardized.columns)
print(len(df_standardized.columns))
print("----")
if _pca == "True":
    df_x_standardized_4_PCA = df_standardized.drop(columns = task_dict[task][0])
    pca_model = pca()
    out = pca_model.fit_transform(df_x_standardized_4_PCA)
    print(out['topfeat'])
    # print(task)
    # pca_model.plot()
    # out['topfeat']
    feature_name_aft_compression = []
    for idx, row in out['topfeat'].iterrows():
        if row["type"] == "best":
            if row["feature"] not in feature_name_aft_compression:
                feature_name_aft_compression.append(row["feature"])
                if pca_truncation == "True" and row["feature"] == "HypoF":
                    break
    print(len(feature_name_aft_compression))
    print(feature_name_aft_compression)
    df_x_aft_PCA_drop = df_x_standardized_4_PCA.drop(df_x_standardized_4_PCA.columns.difference(feature_name_aft_compression), axis = 1)
    df_PCA_ed = pd.concat([df_standardized[task_dict[task][0]], df_x_aft_PCA_drop], axis=1, join='inner')

    print(len(df_PCA_ed.columns))

    df_standardized = df_PCA_ed
df_standardized.head(3)
df_standardized.to_csv(f'data/{task}_PCA_{_pca}_truncate_{pca_truncation}.csv', index=False)

# Binary Classification (task1,2,5,6)
### logistic regression

In [ ]:
import pandas as pd

df_for_logistic_reg = pd.read_csv(f'data/{task}_PCA_{_pca}_truncate_{pca_truncation}.csv')
df_for_logistic_reg.head()
suffled = df_for_logistic_reg.sample(frac=1, random_state=seed).reset_index(drop=True)
# print(len(suffled.columns))
# print(len(suffled))
suffled.corr()
y_train, X_train = suffled.iloc[:, 0], suffled.iloc[:, 1:]
print(len(X_train.columns))

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=len(y_train), n_features=len(X_train.columns),
                           n_informative=3, n_redundant=0,
                           random_state=0, shuffle=False)
clf = AdaBoostClassifier(n_estimators=len(X_train.columns), random_state=0)
clf.fit(X_train, y_train)
clf.score(X_train, y_train)
print(f'model score on training data: {clf.score(X_train, y_train)}')
print(clf.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   clf.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train.columns
)
gini.sort_values(by=["Mean Decrease in Impurity (MDI)"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)
if 'time_to_rem' in gini.index:
    gini.drop(index='time_to_rem', inplace=True)


# Drop 0
n_features = gini.shape[0]
gini = gini[gini['Mean Decrease in Impurity (MDI)'] != 0]

ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14, legend = None)
plt.title(f'AdaBoost Classifier, R2 Score = {clf.score(X_train, y_train)}\n (Number of Features = {n_features+2})', fontsize = 18)
# plt.legend(loc='lower right', fontsize = 12)
plt.axvline(x=0, color='.5')
bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])
plt.subplots_adjust(left=.3)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification
# y_train_, X_train_ = y_train[32:], X_train[32:] # task 1
y_train_, X_train_ = y_train[32:], X_train[32:] # task 2


X, y = make_classification(n_samples=len(y_train_), n_features=len(X_train_.columns),
                           n_informative=3, n_redundant=0,
                           random_state=0, shuffle=False)
clf = AdaBoostClassifier(n_estimators=len(X_train.columns), random_state=0)
clf.fit(X_train_, y_train_)
clf.score(X_train_, y_train_)
print(f'model score on training data: {clf.score(X_train_, y_train_)}')
print(clf.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   clf.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train_.columns
)
gini.sort_values(by=["Mean Decrease in Impurity (MDI)"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)
if 'time_to_rem' in gini.index:
    gini.drop(index='time_to_rem', inplace=True)


# Drop 0
n_features = gini.shape[0]
gini = gini[gini['Mean Decrease in Impurity (MDI)'] != 0]

ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14, legend = None)
score = round(clf.score(X_train, y_train), 3)
plt.title(f'AdaBoost Classifier, R2 Score = {score}\n (Number of Features = {n_features+2})', fontsize = 18)
# plt.legend(loc='lower right', fontsize = 12)
plt.axvline(x=0, color='.5')
bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])
plt.subplots_adjust(left=.3)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification
# y_train__, X_train__ = y_train[25:50], X_train[25:50] # task 1
y_train__, X_train__ = y_train[31:57], X_train[31:57] # task 2


X, y = make_classification(n_samples=len(y_train__), n_features=len(X_train__.columns),
                           n_informative=3, n_redundant=0,
                           random_state=0, shuffle=False)
clf = AdaBoostClassifier(n_estimators=len(X_train__.columns), random_state=0)
clf.fit(X_train__, y_train__)
clf.score(X_train__, y_train__)
print(f'model score on training data: {clf.score(X_train__, y_train__)}')
print(clf.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   clf.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train__.columns
)
gini.sort_values(by=["Mean Decrease in Impurity (MDI)"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)
if 'time_to_rem' in gini.index:
    gini.drop(index='time_to_rem', inplace=True)


# Drop 0
n_features = gini.shape[0]
gini = gini[gini['Mean Decrease in Impurity (MDI)'] != 0]
score = round(clf.score(X_train, y_train), 3)
ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14, legend = None)
plt.title(f'AdaBoost Classifier, R2 Score = {score}\n (Number of Features = {n_features+2})', fontsize = 18)
# plt.legend(loc='lower right', fontsize = 12)
plt.axvline(x=0, color='.5')
bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])
plt.subplots_adjust(left=.3)

### GLM from stats model

In [ ]:
# import numpy as np
# import statsmodels.api as sm
# from scipy import stats
# from matplotlib import pyplot as plt

# plt.rc("figure", figsize=(16,8))
# plt.rc("font", size=14)

In [ ]:
# # glm_ = sm.GLM(y_train, X_train, family=sm.families.Binomial())
# glm_ = sm.GLM(y_train, X_train, family=sm.families.Gamma(sm.families.links.log()))

# glm_.raise_on_perfect_prediction = False
# res = glm_.fit()
# print(res.summary())

In [ ]:
# logit_mod = sm.Logit(y_train, X_train)
# # X_train.corr()
# logit_res = logit_mod.fit()
# print(logit_res.summary())

# Linear Regression (task3,4,8)

In [ ]:
import pandas as pd

df_for_linear_reg = pd.read_csv(f'data/{task}_PCA_{_pca}_truncate_{pca_truncation}.csv')
df_for_linear_reg.head()
suffled = df_for_linear_reg.sample(frac=1, random_state=seed).reset_index(drop=True)
# print(len(suffled.columns))
y_train, X_train = suffled.iloc[:, 0], suffled.iloc[:, 1:]
print(len(X_train.columns))

AdaBoostRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
seed = 2022
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

model = AdaBoostRegressor()
model.fit(X_train, y_train)
model.score(X_train, y_train)
print(f'model score on training data: {model.score(X_train, y_train)}')
print(model.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   model.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train.columns
)
gini = gini.rename(columns={"Mean Decrease in Impurity (MDI)": "MDI"})
gini.sort_values(by=["MDI"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)

# Drop 0
n_features = gini.shape[0]
gini = gini[gini['MDI'] != 0]


# bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
# for i, bar in enumerate(ax.patches):
#     bar.set_color(bar_color[i])
# plt.subplots_adjust(left=.3)



ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14)
plt.title(f'AdaBoost Regressor, R2 Score = {round(model.score(X_train, y_train), 3)}\n (Number of Features = {n_features+1})' , fontsize = 18)
plt.legend(loc='lower right', fontsize = 15)
plt.axvline(x=0, color='.5')

bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])

plt.subplots_adjust(left=.3)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
seed = 2022
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

X_train_, y_train_ = X_train[5:45], y_train[5:45]

model = AdaBoostRegressor()
model.fit(X_train_, y_train_)
model.score(X_train_, y_train_)
print(f'model score on training data: {model.score(X_train_, y_train_)}')
print(model.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   model.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train_.columns
)
gini = gini.rename(columns={"Mean Decrease in Impurity (MDI)": "MDI"})
gini.sort_values(by=["MDI"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)

# Drop 0
n_features = gini.shape[0]
gini = gini[gini['MDI'] != 0]


# bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
# for i, bar in enumerate(ax.patches):
#     bar.set_color(bar_color[i])
# plt.subplots_adjust(left=.3)



ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14)
plt.title(f'AdaBoost Regressor, R2 Score = {round(model.score(X_train, y_train), 3)}\n (Number of Features = {n_features+1})' , fontsize = 18)
plt.legend(loc='lower right', fontsize = 15)
plt.axvline(x=0, color='.5')

bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])

plt.subplots_adjust(left=.3)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
seed = 2022
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

X_train__, y_train__ = X_train[10:55], y_train[10:55]

model = AdaBoostRegressor()
model.fit(X_train__, y_train__)
model.score(X_train__, y_train__)
print(f'model score on training data: {model.score(X_train__, y_train__)}')
print(model.feature_importances_) # Gini importance, Gini Importance or Mean Decrease in Impurity (MDI) calculates each feature importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

gini = pd.DataFrame(
   model.feature_importances_,
   columns=['Mean Decrease in Impurity (MDI)'], index=X_train__.columns
)
gini = gini.rename(columns={"Mean Decrease in Impurity (MDI)": "MDI"})
gini.sort_values(by=["MDI"], inplace = True)
print(gini)
# Drop 'recur' if it exists in the index
if 'recur' in gini.index:
    gini.drop(index='recur', inplace=True)
if 'fu_period' in gini.index:
    gini.drop(index='fu_period', inplace=True)

# Drop 0
n_features = gini.shape[0]
gini = gini[gini['MDI'] != 0]


# bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
# for i, bar in enumerate(ax.patches):
#     bar.set_color(bar_color[i])
# plt.subplots_adjust(left=.3)



ax = gini.plot(kind='barh', figsize=(9, 7), fontsize = 14)
plt.title(f'AdaBoost Regressor, R2 Score = {round(model.score(X_train, y_train), 3)}\n (Number of Features = {n_features+1})' , fontsize = 18)
plt.legend(loc='lower right', fontsize = 15)
plt.axvline(x=0, color='.5')

bar_color = ['#e01e5a' if label == 'ASHS-LIA' else '#236AA7' for label in gini.index]
for i, bar in enumerate(ax.patches):
    bar.set_color(bar_color[i])

plt.subplots_adjust(left=.3)

### LRM from stats model

In [ ]:
# from statsmodels.regression.rolling import RollingWLS
# import numpy as np
# import statsmodels.api as sm
# mod = sm.WLS(y_train, X_train)
# res = mod.fit()
# print(res.summary())
